In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
from pathlib import Path
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

print("\nFiles inside /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    for file in files[:10]:
        print(f"{indent}  {file}")

set correct path

In [ ]:
from pathlib import Path
import os, shutil, json

INPUT_ROOT = Path("/kaggle/input/datasets/maheshakn/research-project-2-small-object-dataset")
WORK_DIR = Path("/kaggle/working/small_object_project")

DATASETS_DIR = WORK_DIR / "datasets"
RUNS_DIR = WORK_DIR / "runs"

VISDRONE_TRAIN = INPUT_ROOT / "VisDrone2019-DET-train" / "VisDrone2019-DET-train"
VISDRONE_VAL = INPUT_ROOT / "VisDrone2019-DET-val" / "VisDrone2019-DET-val"

COCO_VAL_IMAGES = INPUT_ROOT / "val2017" / "val2017"
COCO_ANN = INPUT_ROOT / "annotations_trainval2017" / "annotations" / "instances_val2017.json"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("VISDRONE_TRAIN exists:", VISDRONE_TRAIN.exists())
print("VISDRONE_VAL exists:", VISDRONE_VAL.exists())
print("COCO_VAL_IMAGES exists:", COCO_VAL_IMAGES.exists())
print("COCO_ANN exists:", COCO_ANN.exists())

print("\nTrain images:", len(list((VISDRONE_TRAIN / "images").glob("*.jpg"))))
print("Train annotations:", len(list((VISDRONE_TRAIN / "annotations").glob("*.txt"))))
print("Val images:", len(list((VISDRONE_VAL / "images").glob("*.jpg"))))
print("Val annotations:", len(list((VISDRONE_VAL / "annotations").glob("*.txt"))))
print("COCO val images:", len(list(COCO_VAL_IMAGES.glob("*.jpg"))))

##Cell 3: Convert VisDrone to YOLO format

In [ ]:
from pathlib import Path
from PIL import Image
import shutil
from collections import Counter

OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

splits = {
    "train": VISDRONE_TRAIN,
    "val": VISDRONE_VAL,
}

# VisDrone class IDs:
# 0 ignored, 1 pedestrian, 2 people, 3 bicycle, 4 car, 5 van,
# 6 truck, 7 tricycle, 8 awning-tricycle, 9 bus, 10 motor, 11 others

visdrone_to_common = {
    1: 0,   # pedestrian -> person
    2: 0,   # people -> person
    3: 1,   # bicycle -> bicycle
    4: 2,   # car -> car
    10: 3,  # motor -> motorcycle
    9: 4,   # bus -> bus
    6: 5,   # truck -> truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

def convert_split(split_name, split_path):
    image_dir = split_path / "images"
    ann_dir = split_path / "annotations"

    out_img_dir = OUT_ROOT / "images" / split_name
    out_lbl_dir = OUT_ROOT / "labels" / split_name

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(image_dir.glob("*.jpg"))

    total_boxes = 0
    class_counter = Counter()
    empty_labels = 0

    print(f"Converting {split_name}: {len(image_files)} images")

    for img_path in image_files:
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        ann_path = ann_dir / f"{img_path.stem}.txt"
        out_label_path = out_lbl_dir / f"{img_path.stem}.txt"

        yolo_lines = []

        if ann_path.exists():
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split(",")
                    if len(parts) < 6:
                        continue

                    x, y, w, h = map(float, parts[:4])
                    score = int(float(parts[4]))
                    category = int(float(parts[5]))

                    # Ignore ignored regions
                    if score == 0:
                        continue

                    # Keep only common classes
                    if category not in visdrone_to_common:
                        continue

                    common_class = visdrone_to_common[category]

                    x_center = (x + w / 2) / img_w
                    y_center = (y + h / 2) / img_h
                    norm_w = w / img_w
                    norm_h = h / img_h

                    # Safety clipping
                    if norm_w <= 0 or norm_h <= 0:
                        continue

                    x_center = min(max(x_center, 0), 1)
                    y_center = min(max(y_center, 0), 1)
                    norm_w = min(max(norm_w, 0), 1)
                    norm_h = min(max(norm_h, 0), 1)

                    yolo_lines.append(
                        f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
                    )

                    total_boxes += 1
                    class_counter[common_class] += 1

        shutil.copy(img_path, out_img_dir / img_path.name)

        with open(out_label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        if len(yolo_lines) == 0:
            empty_labels += 1

    print(f"Finished {split_name}")
    print(f"Images: {len(image_files)}")
    print(f"Boxes kept: {total_boxes}")
    print(f"Images with no kept labels: {empty_labels}")
    print("Class distribution:")
    for cls_id, count in sorted(class_counter.items()):
        print(f"  {cls_id} {common_names[cls_id]}: {count}")

for split_name, split_path in splits.items():
    convert_split(split_name, split_path)

print("\nVisDrone YOLO conversion completed.")

#C****ell 4: Create VisDrone YAML file****

In [ ]:
visdrone_yaml = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

yaml_path = OUT_ROOT / "visdrone.yaml"

with open(yaml_path, "w") as f:
    f.write(visdrone_yaml)

print(yaml_path)
print(visdrone_yaml)

Ce****ll 5: Check converted VisDrone dataset****

In [ ]:
base = DATASETS_DIR / "visdrone_yolo"

print("Train images:", len(list((base / "images/train").glob("*.jpg"))))
print("Train labels:", len(list((base / "labels/train").glob("*.txt"))))
print("Val images:", len(list((base / "images/val").glob("*.jpg"))))
print("Val labels:", len(list((base / "labels/val").glob("*.txt"))))

sample_label = next((base / "labels/train").glob("*.txt"))
print("\nSample label file:", sample_label)
print(sample_label.read_text()[:500])

Ce****ll 6: Install Ultralytics YOLOv8****

In [ ]:
!pip install ultralytics -q

In [ ]:
import ultralytics
ultralytics.checks()

Ce****ll 7: Train baseline model E0****

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
epochs=50 \
batch=16 \
workers=2 \
project=/kaggle/working/small_object_project/runs \
name=E0_visdrone_baseline_640

Ce****ll 8: Validate E0 on VisDrone validation set****

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E0_visdrone_baseline_640/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E0_visdrone_val_result

Ce****ll 9: Create COCO-small subset****

In [ ]:
import json
import shutil
from pathlib import Path
from collections import Counter

COCO_OUT_ROOT = DATASETS_DIR / "coco_small_yolo"
OUT_IMG_DIR = COCO_OUT_ROOT / "images" / "val"
OUT_LBL_DIR = COCO_OUT_ROOT / "labels" / "val"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_LBL_DIR.mkdir(parents=True, exist_ok=True)

# COCO category IDs:
# person=1, bicycle=2, car=3, motorcycle=4, bus=6, truck=8

coco_to_common = {
    1: 0,  # person
    2: 1,  # bicycle
    3: 2,  # car
    4: 3,  # motorcycle
    6: 4,  # bus
    8: 5,  # truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

SMALL_AREA_THRESHOLD = 32 * 32

with open(COCO_ANN, "r") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}
anns_by_image = {}

for ann in coco["annotations"]:
    category_id = ann["category_id"]

    if category_id not in coco_to_common:
        continue

    x, y, w, h = ann["bbox"]
    area = w * h

    # COCO small object rule: area < 32^2 pixels
    if area >= SMALL_AREA_THRESHOLD:
        continue

    if w <= 0 or h <= 0:
        continue

    anns_by_image.setdefault(ann["image_id"], []).append(ann)

used_images = 0
used_objects = 0
class_counter = Counter()

for image_id, anns in anns_by_image.items():
    img_info = images[image_id]
    file_name = img_info["file_name"]
    img_w = img_info["width"]
    img_h = img_info["height"]

    src_img = COCO_VAL_IMAGES / file_name
    dst_img = OUT_IMG_DIR / file_name
    dst_lbl = OUT_LBL_DIR / f"{Path(file_name).stem}.txt"

    if not src_img.exists():
        continue

    yolo_lines = []

    for ann in anns:
        x, y, w, h = ann["bbox"]
        common_class = coco_to_common[ann["category_id"]]

        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        if norm_w <= 0 or norm_h <= 0:
            continue

        x_center = min(max(x_center, 0), 1)
        y_center = min(max(y_center, 0), 1)
        norm_w = min(max(norm_w, 0), 1)
        norm_h = min(max(norm_h, 0), 1)

        yolo_lines.append(
            f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
        )

        used_objects += 1
        class_counter[common_class] += 1

    if len(yolo_lines) == 0:
        continue

    shutil.copy(src_img, dst_img)

    with open(dst_lbl, "w") as f:
        f.write("\n".join(yolo_lines))

    used_images += 1

print("COCO-small subset created.")
print("Images used:", used_images)
print("Objects used:", used_objects)
print("Class distribution:")
for cls_id, count in sorted(class_counter.items()):
    print(f"  {cls_id} {common_names[cls_id]}: {count}")

Ce****ll 10: Create COCO-small YAML file****

In [ ]:
coco_small_yaml = f"""
path: {COCO_OUT_ROOT}
train: images/val
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

coco_yaml_path = COCO_OUT_ROOT / "coco_small.yaml"

with open(coco_yaml_path, "w") as f:
    f.write(coco_small_yaml)

print(coco_yaml_path)
print(coco_small_yaml)

Ce****ll 11: Validate E0 model on COCO-small****

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E0_visdrone_baseline_640/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E0_coco_small_domain_shift

E1****: YOLOv8n, image size 960****

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
epochs=30 \
batch=16 \
workers=2 \
project=/kaggle/working/small_object_project/runs \
name=E1_visdrone_resolution_960

In [ ]:
import shutil

output_zip = "/kaggle/working/E0_small_object_results"
shutil.make_archive(output_zip, "zip", "/kaggle/working/small_object_project/runs")

print("Download this file:", output_zip + ".zip")

In [ ]:
import shutil

dataset_zip = "/kaggle/working/converted_yolo_datasets"
shutil.make_archive(dataset_zip, "zip", "/kaggle/working/small_object_project/datasets")

print("Download this file:", dataset_zip + ".zip")

In [ ]:
import os
from pathlib import Path
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

print("\nFiles inside /kaggle/input:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    for file in files[:5]:
        print(f"{indent}  {file}")

In [ ]:
from pathlib import Path
import os, shutil, json

INPUT_ROOT = Path("/kaggle/input/datasets/maheshakn/research-project-2-small-object-dataset")
WORK_DIR = Path("/kaggle/working/small_object_project")

DATASETS_DIR = WORK_DIR / "datasets"
RUNS_DIR = WORK_DIR / "runs"

VISDRONE_TRAIN = INPUT_ROOT / "VisDrone2019-DET-train" / "VisDrone2019-DET-train"
VISDRONE_VAL = INPUT_ROOT / "VisDrone2019-DET-val" / "VisDrone2019-DET-val"

COCO_VAL_IMAGES = INPUT_ROOT / "val2017" / "val2017"
COCO_ANN = INPUT_ROOT / "annotations_trainval2017" / "annotations" / "instances_val2017.json"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("VISDRONE_TRAIN exists:", VISDRONE_TRAIN.exists())
print("VISDRONE_VAL exists:", VISDRONE_VAL.exists())
print("COCO_VAL_IMAGES exists:", COCO_VAL_IMAGES.exists())
print("COCO_ANN exists:", COCO_ANN.exists())

print("\nTrain images:", len(list((VISDRONE_TRAIN / "images").glob("*.jpg"))))
print("Train annotations:", len(list((VISDRONE_TRAIN / "annotations").glob("*.txt"))))
print("Val images:", len(list((VISDRONE_VAL / "images").glob("*.jpg"))))
print("Val annotations:", len(list((VISDRONE_VAL / "annotations").glob("*.txt"))))
print("COCO val images:", len(list(COCO_VAL_IMAGES.glob("*.jpg"))))

In [ ]:
from pathlib import Path

check_path = Path("/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml")
print("Converted VisDrone dataset exists:", check_path.exists())

In [ ]:
from pathlib import Path
import os, shutil, json

INPUT_ROOT = Path("/kaggle/input/datasets/maheshakn/research-project-2-small-object-dataset")
WORK_DIR = Path("/kaggle/working/small_object_project")

DATASETS_DIR = WORK_DIR / "datasets"
RUNS_DIR = WORK_DIR / "runs"

VISDRONE_TRAIN = INPUT_ROOT / "VisDrone2019-DET-train" / "VisDrone2019-DET-train"
VISDRONE_VAL = INPUT_ROOT / "VisDrone2019-DET-val" / "VisDrone2019-DET-val"

COCO_VAL_IMAGES = INPUT_ROOT / "val2017" / "val2017"
COCO_ANN = INPUT_ROOT / "annotations_trainval2017" / "annotations" / "instances_val2017.json"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("VISDRONE_TRAIN exists:", VISDRONE_TRAIN.exists())
print("VISDRONE_VAL exists:", VISDRONE_VAL.exists())
print("COCO_VAL_IMAGES exists:", COCO_VAL_IMAGES.exists())
print("COCO_ANN exists:", COCO_ANN.exists())

print("\nTrain images:", len(list((VISDRONE_TRAIN / "images").glob("*.jpg"))))
print("Train annotations:", len(list((VISDRONE_TRAIN / "annotations").glob("*.txt"))))
print("Val images:", len(list((VISDRONE_VAL / "images").glob("*.jpg"))))
print("Val annotations:", len(list((VISDRONE_VAL / "annotations").glob("*.txt"))))
print("COCO val images:", len(list(COCO_VAL_IMAGES.glob("*.jpg"))))

In [ ]:
from pathlib import Path
from PIL import Image
import shutil
from collections import Counter

OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

splits = {
    "train": VISDRONE_TRAIN,
    "val": VISDRONE_VAL,
}

visdrone_to_common = {
    1: 0,   # pedestrian -> person
    2: 0,   # people -> person
    3: 1,   # bicycle -> bicycle
    4: 2,   # car -> car
    10: 3,  # motor -> motorcycle
    9: 4,   # bus -> bus
    6: 5,   # truck -> truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

def convert_split(split_name, split_path):
    image_dir = split_path / "images"
    ann_dir = split_path / "annotations"

    out_img_dir = OUT_ROOT / "images" / split_name
    out_lbl_dir = OUT_ROOT / "labels" / split_name

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(image_dir.glob("*.jpg"))

    total_boxes = 0
    class_counter = Counter()
    empty_labels = 0

    print(f"Converting {split_name}: {len(image_files)} images")

    for img_path in image_files:
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        ann_path = ann_dir / f"{img_path.stem}.txt"
        out_label_path = out_lbl_dir / f"{img_path.stem}.txt"

        yolo_lines = []

        if ann_path.exists():
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split(",")
                    if len(parts) < 6:
                        continue

                    x, y, w, h = map(float, parts[:4])
                    score = int(float(parts[4]))
                    category = int(float(parts[5]))

                    if score == 0:
                        continue

                    if category not in visdrone_to_common:
                        continue

                    common_class = visdrone_to_common[category]

                    x_center = (x + w / 2) / img_w
                    y_center = (y + h / 2) / img_h
                    norm_w = w / img_w
                    norm_h = h / img_h

                    if norm_w <= 0 or norm_h <= 0:
                        continue

                    x_center = min(max(x_center, 0), 1)
                    y_center = min(max(y_center, 0), 1)
                    norm_w = min(max(norm_w, 0), 1)
                    norm_h = min(max(norm_h, 0), 1)

                    yolo_lines.append(
                        f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
                    )

                    total_boxes += 1
                    class_counter[common_class] += 1

        shutil.copy(img_path, out_img_dir / img_path.name)

        with open(out_label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        if len(yolo_lines) == 0:
            empty_labels += 1

    print(f"Finished {split_name}")
    print(f"Images: {len(image_files)}")
    print(f"Boxes kept: {total_boxes}")
    print(f"Images with no kept labels: {empty_labels}")
    print("Class distribution:")
    for cls_id, count in sorted(class_counter.items()):
        print(f"  {cls_id} {common_names[cls_id]}: {count}")

for split_name, split_path in splits.items():
    convert_split(split_name, split_path)

print("\nVisDrone YOLO conversion completed.")

In [ ]:
OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

visdrone_yaml = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

yaml_path = OUT_ROOT / "visdrone.yaml"

with open(yaml_path, "w") as f:
    f.write(visdrone_yaml)

print(yaml_path)
print(visdrone_yaml)

In [ ]:
base = DATASETS_DIR / "visdrone_yolo"

print("Train images:", len(list((base / "images/train").glob("*.jpg"))))
print("Train labels:", len(list((base / "labels/train").glob("*.txt"))))
print("Val images:", len(list((base / "images/val").glob("*.jpg"))))
print("Val labels:", len(list((base / "labels/val").glob("*.txt"))))

In [ ]:
!pip install ultralytics -q

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=960 \
epochs=30 \
patience=10 \
batch=8 \
workers=2 \
project=/kaggle/working/small_object_project/runs \
name=E1_resolution_960_ep30

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E1_resolution_960_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=960 \
project=/kaggle/working/small_object_project/runs \
name=E1_visdrone_val_result

Ce****ll H — Create COCO-small again****

In [ ]:
import json
import shutil
from pathlib import Path
from collections import Counter

COCO_OUT_ROOT = DATASETS_DIR / "coco_small_yolo"
OUT_IMG_DIR = COCO_OUT_ROOT / "images" / "val"
OUT_LBL_DIR = COCO_OUT_ROOT / "labels" / "val"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_LBL_DIR.mkdir(parents=True, exist_ok=True)

coco_to_common = {
    1: 0,  # person
    2: 1,  # bicycle
    3: 2,  # car
    4: 3,  # motorcycle
    6: 4,  # bus
    8: 5,  # truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

SMALL_AREA_THRESHOLD = 32 * 32

with open(COCO_ANN, "r") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}
anns_by_image = {}

for ann in coco["annotations"]:
    category_id = ann["category_id"]

    if category_id not in coco_to_common:
        continue

    x, y, w, h = ann["bbox"]
    area = w * h

    if area >= SMALL_AREA_THRESHOLD:
        continue

    if w <= 0 or h <= 0:
        continue

    anns_by_image.setdefault(ann["image_id"], []).append(ann)

used_images = 0
used_objects = 0
class_counter = Counter()

for image_id, anns in anns_by_image.items():
    img_info = images[image_id]
    file_name = img_info["file_name"]
    img_w = img_info["width"]
    img_h = img_info["height"]

    src_img = COCO_VAL_IMAGES / file_name
    dst_img = OUT_IMG_DIR / file_name
    dst_lbl = OUT_LBL_DIR / f"{Path(file_name).stem}.txt"

    if not src_img.exists():
        continue

    yolo_lines = []

    for ann in anns:
        x, y, w, h = ann["bbox"]
        common_class = coco_to_common[ann["category_id"]]

        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        if norm_w <= 0 or norm_h <= 0:
            continue

        x_center = min(max(x_center, 0), 1)
        y_center = min(max(y_center, 0), 1)
        norm_w = min(max(norm_w, 0), 1)
        norm_h = min(max(norm_h, 0), 1)

        yolo_lines.append(
            f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
        )

        used_objects += 1
        class_counter[common_class] += 1

    if len(yolo_lines) == 0:
        continue

    shutil.copy(src_img, dst_img)

    with open(dst_lbl, "w") as f:
        f.write("\n".join(yolo_lines))

    used_images += 1

print("COCO-small subset created.")
print("Images used:", used_images)
print("Objects used:", used_objects)
print("Class distribution:")
for cls_id, count in sorted(class_counter.items()):
    print(f"  {cls_id} {common_names[cls_id]}: {count}")

In [ ]:
coco_small_yaml = f"""
path: {COCO_OUT_ROOT}
train: images/val
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

coco_yaml_path = COCO_OUT_ROOT / "coco_small.yaml"

with open(coco_yaml_path, "w") as f:
    f.write(coco_small_yaml)

print(coco_yaml_path)
print(coco_small_yaml)

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E1_resolution_960_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml \
imgsz=960 \
project=/kaggle/working/small_object_project/runs \
name=E1_coco_small_domain_shift

In [ ]:
from pathlib import Path

paths = {
    "VisDrone YAML": Path("/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml"),
    "COCO-small YAML": Path("/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml"),
    "E1 best model": Path("/kaggle/working/small_object_project/runs/E1_resolution_960_ep30/weights/best.pt")
}

for name, path in paths.items():
    print(name, "exists:", path.exists())

In [ ]:
from pathlib import Path
import os, shutil, json

INPUT_ROOT = Path("/kaggle/input/datasets/maheshakn/research-project-2-small-object-dataset")
WORK_DIR = Path("/kaggle/working/small_object_project")

DATASETS_DIR = WORK_DIR / "datasets"
RUNS_DIR = WORK_DIR / "runs"

VISDRONE_TRAIN = INPUT_ROOT / "VisDrone2019-DET-train" / "VisDrone2019-DET-train"
VISDRONE_VAL = INPUT_ROOT / "VisDrone2019-DET-val" / "VisDrone2019-DET-val"

COCO_VAL_IMAGES = INPUT_ROOT / "val2017" / "val2017"
COCO_ANN = INPUT_ROOT / "annotations_trainval2017" / "annotations" / "instances_val2017.json"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("VISDRONE_TRAIN exists:", VISDRONE_TRAIN.exists())
print("VISDRONE_VAL exists:", VISDRONE_VAL.exists())
print("COCO_VAL_IMAGES exists:", COCO_VAL_IMAGES.exists())
print("COCO_ANN exists:", COCO_ANN.exists())

In [ ]:
from pathlib import Path
from PIL import Image
import shutil
from collections import Counter

OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

splits = {
    "train": VISDRONE_TRAIN,
    "val": VISDRONE_VAL,
}

visdrone_to_common = {
    1: 0,   # pedestrian -> person
    2: 0,   # people -> person
    3: 1,   # bicycle -> bicycle
    4: 2,   # car -> car
    10: 3,  # motor -> motorcycle
    9: 4,   # bus -> bus
    6: 5,   # truck -> truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

def convert_split(split_name, split_path):
    image_dir = split_path / "images"
    ann_dir = split_path / "annotations"

    out_img_dir = OUT_ROOT / "images" / split_name
    out_lbl_dir = OUT_ROOT / "labels" / split_name

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(image_dir.glob("*.jpg"))

    total_boxes = 0
    class_counter = Counter()
    empty_labels = 0

    print(f"Converting {split_name}: {len(image_files)} images")

    for img_path in image_files:
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        ann_path = ann_dir / f"{img_path.stem}.txt"
        out_label_path = out_lbl_dir / f"{img_path.stem}.txt"

        yolo_lines = []

        if ann_path.exists():
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split(",")
                    if len(parts) < 6:
                        continue

                    x, y, w, h = map(float, parts[:4])
                    score = int(float(parts[4]))
                    category = int(float(parts[5]))

                    if score == 0:
                        continue

                    if category not in visdrone_to_common:
                        continue

                    common_class = visdrone_to_common[category]

                    x_center = (x + w / 2) / img_w
                    y_center = (y + h / 2) / img_h
                    norm_w = w / img_w
                    norm_h = h / img_h

                    if norm_w <= 0 or norm_h <= 0:
                        continue

                    x_center = min(max(x_center, 0), 1)
                    y_center = min(max(y_center, 0), 1)
                    norm_w = min(max(norm_w, 0), 1)
                    norm_h = min(max(norm_h, 0), 1)

                    yolo_lines.append(
                        f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
                    )

                    total_boxes += 1
                    class_counter[common_class] += 1

        shutil.copy(img_path, out_img_dir / img_path.name)

        with open(out_label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        if len(yolo_lines) == 0:
            empty_labels += 1

    print(f"Finished {split_name}")
    print(f"Images: {len(image_files)}")
    print(f"Boxes kept: {total_boxes}")
    print(f"Images with no kept labels: {empty_labels}")
    print("Class distribution:")
    for cls_id, count in sorted(class_counter.items()):
        print(f"  {cls_id} {common_names[cls_id]}: {count}")

for split_name, split_path in splits.items():
    convert_split(split_name, split_path)

print("\nVisDrone YOLO conversion completed.")

In [ ]:
OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

visdrone_yaml = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

yaml_path = OUT_ROOT / "visdrone.yaml"

with open(yaml_path, "w") as f:
    f.write(visdrone_yaml)

print(yaml_path)
print(visdrone_yaml)

In [ ]:
base = DATASETS_DIR / "visdrone_yolo"

print("Train images:", len(list((base / "images/train").glob("*.jpg"))))
print("Train labels:", len(list((base / "labels/train").glob("*.txt"))))
print("Val images:", len(list((base / "images/val").glob("*.jpg"))))
print("Val labels:", len(list((base / "labels/val").glob("*.txt"))))

In [ ]:
import json
import shutil
from pathlib import Path
from collections import Counter

COCO_OUT_ROOT = DATASETS_DIR / "coco_small_yolo"
OUT_IMG_DIR = COCO_OUT_ROOT / "images" / "val"
OUT_LBL_DIR = COCO_OUT_ROOT / "labels" / "val"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_LBL_DIR.mkdir(parents=True, exist_ok=True)

coco_to_common = {
    1: 0,  # person
    2: 1,  # bicycle
    3: 2,  # car
    4: 3,  # motorcycle
    6: 4,  # bus
    8: 5,  # truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

SMALL_AREA_THRESHOLD = 32 * 32

with open(COCO_ANN, "r") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}
anns_by_image = {}

for ann in coco["annotations"]:
    category_id = ann["category_id"]

    if category_id not in coco_to_common:
        continue

    x, y, w, h = ann["bbox"]
    area = w * h

    if area >= SMALL_AREA_THRESHOLD:
        continue

    if w <= 0 or h <= 0:
        continue

    anns_by_image.setdefault(ann["image_id"], []).append(ann)

used_images = 0
used_objects = 0
class_counter = Counter()

for image_id, anns in anns_by_image.items():
    img_info = images[image_id]
    file_name = img_info["file_name"]
    img_w = img_info["width"]
    img_h = img_info["height"]

    src_img = COCO_VAL_IMAGES / file_name
    dst_img = OUT_IMG_DIR / file_name
    dst_lbl = OUT_LBL_DIR / f"{Path(file_name).stem}.txt"

    if not src_img.exists():
        continue

    yolo_lines = []

    for ann in anns:
        x, y, w, h = ann["bbox"]
        common_class = coco_to_common[ann["category_id"]]

        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        if norm_w <= 0 or norm_h <= 0:
            continue

        x_center = min(max(x_center, 0), 1)
        y_center = min(max(y_center, 0), 1)
        norm_w = min(max(norm_w, 0), 1)
        norm_h = min(max(norm_h, 0), 1)

        yolo_lines.append(
            f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
        )

        used_objects += 1
        class_counter[common_class] += 1

    if len(yolo_lines) == 0:
        continue

    shutil.copy(src_img, dst_img)

    with open(dst_lbl, "w") as f:
        f.write("\n".join(yolo_lines))

    used_images += 1

print("COCO-small subset created.")
print("Images used:", used_images)
print("Objects used:", used_objects)
print("Class distribution:")
for cls_id, count in sorted(class_counter.items()):
    print(f"  {cls_id} {common_names[cls_id]}: {count}")

In [ ]:
coco_small_yaml = f"""
path: {COCO_OUT_ROOT}
train: images/val
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

coco_yaml_path = COCO_OUT_ROOT / "coco_small.yaml"

with open(coco_yaml_path, "w") as f:
    f.write(coco_small_yaml)

print(coco_yaml_path)
print(coco_small_yaml)

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
epochs=30 \
patience=10 \
batch=8 \
workers=2 \
mosaic=0.0 \
project=/kaggle/working/small_object_project/runs \
name=E2_mosaic_disabled_ep30

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E2_mosaic_disabled_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E2_visdrone_val_result

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E2_mosaic_disabled_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E2_coco_small_domain_shift

E3**** appearance augmentation****

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
epochs=30 \
patience=10 \
batch=8 \
workers=2 \
hsv_h=0.015 hsv_s=0.7 hsv_v=0.5 \
project=/kaggle/working/small_object_project/runs \
name=E3_appearance_aug_ep30

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E3_appearance_aug_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E3_visdrone_val_result

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E3_appearance_aug_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E3_coco_small_domain_shift

E4**** scale augmentation****

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
epochs=30 \
patience=10 \
batch=8 \
workers=2 \
scale=0.7 \
project=/kaggle/working/small_object_project/runs \
name=E4_scale_aug_ep30

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E4_scale_aug_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E4_visdrone_val_result

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E4_scale_aug_ep30/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml \
imgsz=640 \
project=/kaggle/working/small_object_project/runs \
name=E4_coco_small_domain_shift

In [ ]:
!pip install ultralytics -q

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
from pathlib import Path

print(Path("/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml").exists())
print(Path("/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml").exists())

In [ ]:
from pathlib import Path
import os

print("Files/folders in /kaggle/working:")
for p in Path("/kaggle/working").iterdir():
    print(p)

In [ ]:
from pathlib import Path

runs_path = Path("/kaggle/working/small_object_project/runs")
print("Runs folder exists:", runs_path.exists())

ir****st generate your final result graphs****

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/report_graphs")
OUT.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame({
    "Experiment": [
        "E0 Baseline",
        "E1 Resolution 960",
        "E2 Mosaic Disabled",
        "E3 Appearance Aug",
        "E4 Scale Aug"
    ],
    "VisDrone_mAP50": [0.364, 0.457, 0.347, 0.344, 0.345],
    "VisDrone_mAP50_95": [0.202, 0.270, 0.192, 0.191, 0.191],
    "COCO_mAP50": [0.0276, 0.0493, 0.0280, 0.0255, 0.0245],
    "COCO_mAP50_95": [0.0105, 0.0219, 0.0106, 0.00939, 0.00968],
})

df["Domain_Gap"] = df["VisDrone_mAP50_95"] - df["COCO_mAP50_95"]

display(df)

# Graph 1: mAP50-95 comparison
plt.figure(figsize=(10, 5))
x = range(len(df))
plt.bar([i - 0.2 for i in x], df["VisDrone_mAP50_95"], width=0.4, label="VisDrone mAP50-95")
plt.bar([i + 0.2 for i in x], df["COCO_mAP50_95"], width=0.4, label="COCO-small mAP50-95")
plt.xticks(x, df["Experiment"], rotation=25, ha="right")
plt.ylabel("mAP50-95")
plt.title("In-domain vs Out-of-domain Performance")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "map50_95_comparison.png", dpi=300)
plt.show()

# Graph 2: Domain gap
plt.figure(figsize=(10, 5))
plt.bar(df["Experiment"], df["Domain_Gap"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Domain gap")
plt.title("Domain Gap Across Experiments")
plt.tight_layout()
plt.savefig(OUT / "domain_gap.png", dpi=300)
plt.show()

# Graph 3: mAP50 comparison
plt.figure(figsize=(10, 5))
plt.bar([i - 0.2 for i in x], df["VisDrone_mAP50"], width=0.4, label="VisDrone mAP50")
plt.bar([i + 0.2 for i in x], df["COCO_mAP50"], width=0.4, label="COCO-small mAP50")
plt.xticks(x, df["Experiment"], rotation=25, ha="right")
plt.ylabel("mAP50")
plt.title("mAP50 Comparison Across Experiments")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "map50_comparison.png", dpi=300)
plt.show()

print("Graphs saved in:", OUT)

Th****en create class distribution graphs****

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("/kaggle/working/report_graphs")
OUT.mkdir(parents=True, exist_ok=True)

classes = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

visdrone_train = [106396, 10480, 144866, 29647, 5926, 12875]
visdrone_val = [13969, 1287, 14064, 4886, 251, 750]
coco_small = [3217, 88, 1005, 54, 26, 81]

plt.figure(figsize=(10, 5))
plt.bar(classes, visdrone_train)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of objects")
plt.title("VisDrone Train Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "visdrone_train_class_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(classes, visdrone_val)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of objects")
plt.title("VisDrone Validation Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "visdrone_val_class_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(classes, coco_small)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of objects")
plt.title("COCO-small Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "coco_small_class_distribution.png", dpi=300)
plt.show()

In [ ]:
import shutil

graph_zip = "/kaggle/working/report_graphs"
shutil.make_archive(graph_zip, "zip", "/kaggle/working/report_graphs")

print("Download this file:", graph_zip + ".zip")

In [ ]:
from pathlib import Path
import os, shutil, json

INPUT_ROOT = Path("/kaggle/input/datasets/maheshakn/research-project-2-small-object-dataset")
WORK_DIR = Path("/kaggle/working/small_object_project")

DATASETS_DIR = WORK_DIR / "datasets"
RUNS_DIR = WORK_DIR / "runs"

VISDRONE_TRAIN = INPUT_ROOT / "VisDrone2019-DET-train" / "VisDrone2019-DET-train"
VISDRONE_VAL = INPUT_ROOT / "VisDrone2019-DET-val" / "VisDrone2019-DET-val"

COCO_VAL_IMAGES = INPUT_ROOT / "val2017" / "val2017"
COCO_ANN = INPUT_ROOT / "annotations_trainval2017" / "annotations" / "instances_val2017.json"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("VISDRONE_TRAIN exists:", VISDRONE_TRAIN.exists())
print("VISDRONE_VAL exists:", VISDRONE_VAL.exists())
print("COCO_VAL_IMAGES exists:", COCO_VAL_IMAGES.exists())
print("COCO_ANN exists:", COCO_ANN.exists())

In [ ]:
from pathlib import Path
from PIL import Image
import shutil
from collections import Counter

OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

splits = {
    "train": VISDRONE_TRAIN,
    "val": VISDRONE_VAL,
}

visdrone_to_common = {
    1: 0,   # pedestrian -> person
    2: 0,   # people -> person
    3: 1,   # bicycle -> bicycle
    4: 2,   # car -> car
    10: 3,  # motor -> motorcycle
    9: 4,   # bus -> bus
    6: 5,   # truck -> truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

def convert_split(split_name, split_path):
    image_dir = split_path / "images"
    ann_dir = split_path / "annotations"

    out_img_dir = OUT_ROOT / "images" / split_name
    out_lbl_dir = OUT_ROOT / "labels" / split_name

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(image_dir.glob("*.jpg"))

    total_boxes = 0
    class_counter = Counter()
    empty_labels = 0

    print(f"Converting {split_name}: {len(image_files)} images")

    for img_path in image_files:
        with Image.open(img_path) as img:
            img_w, img_h = img.size

        ann_path = ann_dir / f"{img_path.stem}.txt"
        out_label_path = out_lbl_dir / f"{img_path.stem}.txt"

        yolo_lines = []

        if ann_path.exists():
            with open(ann_path, "r") as f:
                for line in f:
                    parts = line.strip().split(",")
                    if len(parts) < 6:
                        continue

                    x, y, w, h = map(float, parts[:4])
                    score = int(float(parts[4]))
                    category = int(float(parts[5]))

                    if score == 0:
                        continue

                    if category not in visdrone_to_common:
                        continue

                    common_class = visdrone_to_common[category]

                    x_center = (x + w / 2) / img_w
                    y_center = (y + h / 2) / img_h
                    norm_w = w / img_w
                    norm_h = h / img_h

                    if norm_w <= 0 or norm_h <= 0:
                        continue

                    x_center = min(max(x_center, 0), 1)
                    y_center = min(max(y_center, 0), 1)
                    norm_w = min(max(norm_w, 0), 1)
                    norm_h = min(max(norm_h, 0), 1)

                    yolo_lines.append(
                        f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
                    )

                    total_boxes += 1
                    class_counter[common_class] += 1

        shutil.copy(img_path, out_img_dir / img_path.name)

        with open(out_label_path, "w") as f:
            f.write("\n".join(yolo_lines))

        if len(yolo_lines) == 0:
            empty_labels += 1

    print(f"Finished {split_name}")
    print(f"Images: {len(image_files)}")
    print(f"Boxes kept: {total_boxes}")
    print(f"Images with no kept labels: {empty_labels}")
    print("Class distribution:")
    for cls_id, count in sorted(class_counter.items()):
        print(f"  {cls_id} {common_names[cls_id]}: {count}")

for split_name, split_path in splits.items():
    convert_split(split_name, split_path)

print("\nVisDrone YOLO conversion completed.")

In [ ]:
OUT_ROOT = DATASETS_DIR / "visdrone_yolo"

visdrone_yaml = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

yaml_path = OUT_ROOT / "visdrone.yaml"

with open(yaml_path, "w") as f:
    f.write(visdrone_yaml)

print(yaml_path)
print(visdrone_yaml)

In [ ]:
base = DATASETS_DIR / "visdrone_yolo"

print("Train images:", len(list((base / "images/train").glob("*.jpg"))))
print("Train labels:", len(list((base / "labels/train").glob("*.txt"))))
print("Val images:", len(list((base / "images/val").glob("*.jpg"))))
print("Val labels:", len(list((base / "labels/val").glob("*.txt"))))

In [ ]:
import json
import shutil
from pathlib import Path
from collections import Counter

COCO_OUT_ROOT = DATASETS_DIR / "coco_small_yolo"
OUT_IMG_DIR = COCO_OUT_ROOT / "images" / "val"
OUT_LBL_DIR = COCO_OUT_ROOT / "labels" / "val"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_LBL_DIR.mkdir(parents=True, exist_ok=True)

coco_to_common = {
    1: 0,  # person
    2: 1,  # bicycle
    3: 2,  # car
    4: 3,  # motorcycle
    6: 4,  # bus
    8: 5,  # truck
}

common_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
}

SMALL_AREA_THRESHOLD = 32 * 32

with open(COCO_ANN, "r") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}
anns_by_image = {}

for ann in coco["annotations"]:
    category_id = ann["category_id"]

    if category_id not in coco_to_common:
        continue

    x, y, w, h = ann["bbox"]
    area = w * h

    if area >= SMALL_AREA_THRESHOLD:
        continue

    if w <= 0 or h <= 0:
        continue

    anns_by_image.setdefault(ann["image_id"], []).append(ann)

used_images = 0
used_objects = 0
class_counter = Counter()

for image_id, anns in anns_by_image.items():
    img_info = images[image_id]
    file_name = img_info["file_name"]
    img_w = img_info["width"]
    img_h = img_info["height"]

    src_img = COCO_VAL_IMAGES / file_name
    dst_img = OUT_IMG_DIR / file_name
    dst_lbl = OUT_LBL_DIR / f"{Path(file_name).stem}.txt"

    if not src_img.exists():
        continue

    yolo_lines = []

    for ann in anns:
        x, y, w, h = ann["bbox"]
        common_class = coco_to_common[ann["category_id"]]

        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        if norm_w <= 0 or norm_h <= 0:
            continue

        x_center = min(max(x_center, 0), 1)
        y_center = min(max(y_center, 0), 1)
        norm_w = min(max(norm_w, 0), 1)
        norm_h = min(max(norm_h, 0), 1)

        yolo_lines.append(
            f"{common_class} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
        )

        used_objects += 1
        class_counter[common_class] += 1

    if len(yolo_lines) == 0:
        continue

    shutil.copy(src_img, dst_img)

    with open(dst_lbl, "w") as f:
        f.write("\n".join(yolo_lines))

    used_images += 1

print("COCO-small subset created.")
print("Images used:", used_images)
print("Objects used:", used_objects)
print("Class distribution:")
for cls_id, count in sorted(class_counter.items()):
    print(f"  {cls_id} {common_names[cls_id]}: {count}")

In [ ]:
coco_small_yaml = f"""
path: {COCO_OUT_ROOT}
train: images/val
val: images/val

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

coco_yaml_path = COCO_OUT_ROOT / "coco_small.yaml"

with open(coco_yaml_path, "w") as f:
    f.write(coco_small_yaml)

print(coco_yaml_path)
print(coco_small_yaml)

In [ ]:
!pip install ultralytics -q

In [ ]:
!yolo detect train \
model=yolov8n.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=960 \
epochs=30 \
patience=10 \
batch=8 \
workers=2 \
plots=True \
project=/kaggle/working/small_object_project/runs \
name=E1_resolution_960_ep30_report

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E1_resolution_960_ep30_report/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/visdrone_yolo/visdrone.yaml \
imgsz=960 \
plots=True \
project=/kaggle/working/small_object_project/runs \
name=E1_visdrone_val_report

In [ ]:
!yolo detect val \
model=/kaggle/working/small_object_project/runs/E1_resolution_960_ep30_report/weights/best.pt \
data=/kaggle/working/small_object_project/datasets/coco_small_yolo/coco_small.yaml \
imgsz=960 \
plots=True \
project=/kaggle/working/small_object_project/runs \
name=E1_coco_small_val_report

In [ ]:
!yolo detect predict \
model=/kaggle/working/small_object_project/runs/E1_resolution_960_ep30_report/weights/best.pt \
source=/kaggle/working/small_object_project/datasets/visdrone_yolo/images/val \
imgsz=960 \
conf=0.25 \
save=True \
project=/kaggle/working/small_object_project/runs \
name=E1_visdrone_prediction_examples

In [ ]:
!yolo detect predict \
model=/kaggle/working/small_object_project/runs/E1_resolution_960_ep30_report/weights/best.pt \
source=/kaggle/working/small_object_project/datasets/coco_small_yolo/images/val \
imgsz=960 \
conf=0.25 \
save=True \
project=/kaggle/working/small_object_project/runs \
name=E1_coco_prediction_examples

In [ ]:
from pathlib import Path

runs_dir = Path("/kaggle/working/small_object_project/runs")

for exp in runs_dir.iterdir():
    if exp.is_dir():
        print("\nFolder:", exp.name)
        for f in exp.glob("*"):
            if f.suffix.lower() in [".png", ".jpg", ".jpeg"]:
                print("  ", f.name)

In [ ]:
import shutil

output_zip = "/kaggle/working/E1_report_outputs"
shutil.make_archive(
    output_zip,
    "zip",
    "/kaggle/working/small_object_project/runs"
)

print("Download this file:", output_zip + ".zip")

In [ ]:
from pathlib import Path

zip_path = Path("/kaggle/working/E1_report_outputs.zip")
print("Exists:", zip_path.exists())

if zip_path.exists():
    print("Size MB:", zip_path.stat().st_size / (1024 * 1024))

In [ ]:
from pathlib import Path
import shutil

RUNS = Path("/kaggle/working/small_object_project/runs")
OUT = Path("/kaggle/working/E1_report_selected_files")

if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(parents=True, exist_ok=True)

selected_files = [
    # Training outputs
    "E1_resolution_960_ep30_report/results.png",
    "E1_resolution_960_ep30_report/confusion_matrix.png",
    "E1_resolution_960_ep30_report/confusion_matrix_normalized.png",
    "E1_resolution_960_ep30_report/PR_curve.png",
    "E1_resolution_960_ep30_report/F1_curve.png",
    "E1_resolution_960_ep30_report/P_curve.png",
    "E1_resolution_960_ep30_report/R_curve.png",

    # VisDrone validation outputs
    "E1_visdrone_val_report/confusion_matrix.png",
    "E1_visdrone_val_report/confusion_matrix_normalized.png",
    "E1_visdrone_val_report/PR_curve.png",
    "E1_visdrone_val_report/F1_curve.png",
    "E1_visdrone_val_report/val_batch0_pred.jpg",
    "E1_visdrone_val_report/val_batch0_labels.jpg",
    "E1_visdrone_val_report/val_batch1_pred.jpg",
    "E1_visdrone_val_report/val_batch1_labels.jpg",

    # COCO-small validation outputs
    "E1_coco_small_val_report/confusion_matrix.png",
    "E1_coco_small_val_report/confusion_matrix_normalized.png",
    "E1_coco_small_val_report/PR_curve.png",
    "E1_coco_small_val_report/F1_curve.png",
    "E1_coco_small_val_report/val_batch0_pred.jpg",
    "E1_coco_small_val_report/val_batch0_labels.jpg",
]

copied = 0

for rel_path in selected_files:
    src = RUNS / rel_path
    if src.exists():
        dst = OUT / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        copied += 1
        print("Copied:", rel_path)
    else:
        print("Missing:", rel_path)

print("Total copied:", copied)

In [ ]:
from pathlib import Path
import shutil

prediction_folders = [
    "E1_visdrone_prediction_examples",
    "E1_coco_prediction_examples"
]

for folder in prediction_folders:
    src_folder = RUNS / folder
    dst_folder = OUT / folder
    dst_folder.mkdir(parents=True, exist_ok=True)

    if src_folder.exists():
        image_files = list(src_folder.glob("*.jpg"))[:10]  # copy only first 10 images
        for img in image_files:
            shutil.copy(img, dst_folder / img.name)
        print(folder, "copied images:", len(image_files))
    else:
        print("Missing folder:", folder)

In [ ]:
from pathlib import Path
import shutil
from IPython.display import FileLink

RUNS = Path("/kaggle/working/small_object_project/runs")
OUT = Path("/kaggle/working/latest_report_outputs_selected")

# Clean old folder
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

# Important report files from E1
selected_files = [
    # E1 training outputs
    "E1_resolution_960_ep30_report/results.png",
    "E1_resolution_960_ep30_report/confusion_matrix.png",
    "E1_resolution_960_ep30_report/confusion_matrix_normalized.png",
    "E1_resolution_960_ep30_report/PR_curve.png",
    "E1_resolution_960_ep30_report/F1_curve.png",
    "E1_resolution_960_ep30_report/P_curve.png",
    "E1_resolution_960_ep30_report/R_curve.png",

    # E1 VisDrone validation outputs
    "E1_visdrone_val_report/confusion_matrix.png",
    "E1_visdrone_val_report/confusion_matrix_normalized.png",
    "E1_visdrone_val_report/PR_curve.png",
    "E1_visdrone_val_report/F1_curve.png",
    "E1_visdrone_val_report/P_curve.png",
    "E1_visdrone_val_report/R_curve.png",
    "E1_visdrone_val_report/val_batch0_pred.jpg",
    "E1_visdrone_val_report/val_batch0_labels.jpg",
    "E1_visdrone_val_report/val_batch1_pred.jpg",
    "E1_visdrone_val_report/val_batch1_labels.jpg",

    # E1 COCO-small validation outputs
    "E1_coco_small_val_report/confusion_matrix.png",
    "E1_coco_small_val_report/confusion_matrix_normalized.png",
    "E1_coco_small_val_report/PR_curve.png",
    "E1_coco_small_val_report/F1_curve.png",
    "E1_coco_small_val_report/P_curve.png",
    "E1_coco_small_val_report/R_curve.png",
    "E1_coco_small_val_report/val_batch0_pred.jpg",
    "E1_coco_small_val_report/val_batch0_labels.jpg",
]

copied = 0
missing = []

for rel_path in selected_files:
    src = RUNS / rel_path
    if src.exists():
        dst = OUT / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        copied += 1
        print("Copied:", rel_path)
    else:
        missing.append(rel_path)
        print("Missing:", rel_path)

# Copy only first 10 prediction images from each prediction folder
prediction_folders = [
    "E1_visdrone_prediction_examples",
    "E1_coco_prediction_examples"
]

for folder in prediction_folders:
    src_folder = RUNS / folder
    dst_folder = OUT / folder
    dst_folder.mkdir(parents=True, exist_ok=True)

    if src_folder.exists():
        imgs = sorted(list(src_folder.glob("*.jpg")))[:10]
        for img in imgs:
            shutil.copy(img, dst_folder / img.name)
        print(f"Copied {len(imgs)} images from {folder}")
    else:
        missing.append(folder)
        print("Missing folder:", folder)

# Also create a text file with your E0-E4 numerical results
results_txt = OUT / "E0_to_E4_results_summary.txt"
results_txt.write_text("""
Final Experimental Results Summary

E0 Baseline 640:
VisDrone mAP50 = 0.364
VisDrone mAP50-95 = 0.202
COCO-small mAP50 = 0.0276
COCO-small mAP50-95 = 0.0105
Domain gap = 0.1915

E1 Resolution 960:
VisDrone mAP50 = 0.457
VisDrone mAP50-95 = 0.270
COCO-small mAP50 = 0.0493
COCO-small mAP50-95 = 0.0219
Domain gap = 0.2481

E2 Mosaic Disabled:
VisDrone mAP50 = 0.347
VisDrone mAP50-95 = 0.192
COCO-small mAP50 = 0.0280
COCO-small mAP50-95 = 0.0106
Domain gap = 0.1814

E3 Appearance Augmentation:
VisDrone mAP50 = 0.344
VisDrone mAP50-95 = 0.191
COCO-small mAP50 = 0.0255
COCO-small mAP50-95 = 0.00939
Domain gap = 0.1816

E4 Scale Augmentation:
VisDrone mAP50 = 0.345
VisDrone mAP50-95 = 0.191
COCO-small mAP50 = 0.0245
COCO-small mAP50-95 = 0.00968
Domain gap = 0.1813

Main finding:
E1, higher image resolution 960, achieved the best performance.
""")

print("\nSummary file created:", results_txt)

# Zip folder
zip_base = "/kaggle/working/latest_report_outputs_selected"
zip_path = shutil.make_archive(zip_base, "zip", OUT)

zip_file = Path(zip_path)
print("\nCreated zip:", zip_file)
print("Size MB:", round(zip_file.stat().st_size / (1024 * 1024), 2))
print("Files copied:", copied)
print("Missing items:", len(missing))

# Create clickable download link
FileLink(str(zip_file))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import shutil
from IPython.display import FileLink

OUT = Path("/kaggle/working/final_report_graphs")
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Dataset class distribution
# -----------------------------
classes = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

visdrone_train = [106396, 10480, 144866, 29647, 5926, 12875]
visdrone_val = [13969, 1287, 14064, 4886, 251, 750]
coco_small = [3217, 88, 1005, 54, 26, 81]

plt.figure(figsize=(10, 5))
plt.bar(classes, visdrone_train)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of objects")
plt.title("VisDrone Train Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "visdrone_train_class_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(classes, visdrone_val)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of objects")
plt.title("VisDrone Validation Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "visdrone_val_class_distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(classes, coco_small)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of objects")
plt.title("COCO-small Class Distribution")
plt.tight_layout()
plt.savefig(OUT / "coco_small_class_distribution.png", dpi=300)
plt.show()

# -----------------------------
# E0-E4 result graphs
# -----------------------------
df = pd.DataFrame({
    "Experiment": [
        "E0 Baseline",
        "E1 Resolution 960",
        "E2 Mosaic Disabled",
        "E3 Appearance Aug",
        "E4 Scale Aug"
    ],
    "VisDrone_mAP50": [0.364, 0.457, 0.347, 0.344, 0.345],
    "VisDrone_mAP50_95": [0.202, 0.270, 0.192, 0.191, 0.191],
    "COCO_mAP50": [0.0276, 0.0493, 0.0280, 0.0255, 0.0245],
    "COCO_mAP50_95": [0.0105, 0.0219, 0.0106, 0.00939, 0.00968],
})

df["Domain_Gap"] = df["VisDrone_mAP50_95"] - df["COCO_mAP50_95"]

display(df)

# mAP50-95 comparison
plt.figure(figsize=(10, 5))
x = range(len(df))
plt.bar([i - 0.2 for i in x], df["VisDrone_mAP50_95"], width=0.4, label="VisDrone mAP50-95")
plt.bar([i + 0.2 for i in x], df["COCO_mAP50_95"], width=0.4, label="COCO-small mAP50-95")
plt.xticks(x, df["Experiment"], rotation=25, ha="right")
plt.ylabel("mAP50-95")
plt.title("In-domain vs Out-of-domain Performance")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "map50_95_comparison.png", dpi=300)
plt.show()

# mAP50 comparison
plt.figure(figsize=(10, 5))
plt.bar([i - 0.2 for i in x], df["VisDrone_mAP50"], width=0.4, label="VisDrone mAP50")
plt.bar([i + 0.2 for i in x], df["COCO_mAP50"], width=0.4, label="COCO-small mAP50")
plt.xticks(x, df["Experiment"], rotation=25, ha="right")
plt.ylabel("mAP50")
plt.title("mAP50 Comparison Across Experiments")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "map50_comparison.png", dpi=300)
plt.show()

# Domain gap
plt.figure(figsize=(10, 5))
plt.bar(df["Experiment"], df["Domain_Gap"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Domain gap")
plt.title("Domain Gap Across Experiments")
plt.tight_layout()
plt.savefig(OUT / "domain_gap.png", dpi=300)
plt.show()

# Zip all final graphs
zip_base = "/kaggle/working/final_report_graphs"
shutil.make_archive(zip_base, "zip", OUT)

print("Created:", zip_base + ".zip")
FileLink(zip_base + ".zip")